# BOM

## 01. BOM processor (from single-level to multi-level BOM)


- input: single-level BOM
- task: using recursive production BOM algorithm
- output: multi-level BOM

### base - iterative ways - (ĐÚNG)

In [ ]:
import pandas as pd
from collections import defaultdict

def explode_bom(flat_bom_df, roots, parent_col='parent', comp_col='component', qty_col='qty',
                include_root=True, stop_on_cycle=True):
    """
    Expand a single-level BOM (parent → component) into a multi-level structure.
    
    Parameters
    ----------
    flat_bom_df : pd.DataFrame
        Columns: parent, component, qty
    roots : str | list
        One or more top-level items to explode.
    include_root : bool
        If True, add a level-0 record for each root.
    stop_on_cycle : bool
        If True, raises on BOM cycle.
        
    Returns
    -------
    exploded_df : pd.DataFrame
        Columns: root, parent_path, child, level, qty_per_parent, cumulative_qty, full_path
    """
    # convert single root into a list
    if isinstance(roots, (str, int)):
        roots = [roots]

    df = flat_bom_df.copy()
    
    if qty_col not in df.columns:
        df[qty_col] = 1

    # Build adjacency dictionary: parent → list of (component, qty)
    adj = defaultdict(list)
    for _, row in df.iterrows():
        adj[row[parent_col]].append((row[comp_col], float(row[qty_col])))

    rows = []

    for root in roots:
        # Initialize stack: (node, cumulative_qty, path_list, level, parent, qty)
        stack = [(root, 1.0, [root], 0, None, 1.0)]


        while stack:
            
            node, cum_qty, path, level, parent, qty = stack.pop()
            
            # append WHEN a node is popped, eg: actually explore (skip root if include_root==False)
            
            if not (level == 0 and parent is None and not include_root):
                rows.append({
                    'root': root,
                    'parent_path': '' if parent is None else ' > '.join(path[:-1]),
                    'child': node,
                    'level': level,
                    'qty_per_parent': qty,
                    'cumulative_qty': cum_qty,
                    'full_path': ' > '.join(path)
                })
                
            
            
            # Child processing
            for child, rel_qty in reversed(adj.get(node, []) ): #thay doi chieu xu ly de : thu tu left-to-righh or nguoc lai
                # cycle detection
                if child in path:
                    if stop_on_cycle:
                        raise ValueError(f"Cycle detected: {' > '.join(path + [child])}")
                    continue
                
                # main logic
                new_cum = cum_qty * rel_qty
                new_path = path + [child]

                # Push child first, result will be appended *after* it’s expanded
                # (node, cumulative_qty, path_list, level, parent, qty)
                stack.append((child, new_cum, new_path, level + 1, node, rel_qty))
                




    exploded_df = pd.DataFrame(rows)
    
    return exploded_df


# --- Example ---
if __name__ == "__main__":
    data = [
        ('A', 'B', 2),
        ('A', 'C', 1),
        ('B', 'D', 3),
        ('B', 'E', 2),
        ('C', 'F', 4),
        ('E', 'G', 1)
    ]

    flat_bom = pd.DataFrame(data, columns=['parent', 'component', 'qty'])
    exploded = explode_bom(flat_bom, roots=['A','B'], include_root=True)
    display(exploded)

### study - input & parameter - SAI (keep nha)

In [ ]:
# input data and parameter
data = [
    ('A', 'B', 2),
    ('A', 'C', 1),
    ('B', 'D', 3),
    ('B', 'E', 2),
    ('C', 'F', 4),
    ('E', 'G', 1)
]

flat_bom = pd.DataFrame(data, columns=['parent', 'component', 'qty'])
print(flat_bom)

ROOTS = ["A","B"]
PARENT_COL = "parent"
COM_COL = "component"
QTY_COL = "qty"
INCLUDE_ROOT = True
STOP_ON_CYCLE = True

### study - algorithm - SAI (keep nha)

- nhận xét : order of algorithm = ĐÚNG, but order of print result = SAI
- REASON : state vector of the TASK / STACK là quá ít

In [ ]:
df = flat_bom.copy()

"""0. pre-processing , data validation"""

#--- Programming concept: input normalization or making the code flexible.

# convert single root into a list
if isinstance(ROOTS, (str, int)):
    ROOTS = [ROOTS]
# if no data in col qty, default value is 1 
if QTY_COL not in df.columns:
    df[QTY_COL] = 1



# Build adjacency dictionary: parent → list of (component, qty)
# This is the single-level BOM converted to graph form
# This is your product structure graph: Each edge represents a "contains" relationship with usage quantity
adj = defaultdict(list)
for _, row in df.iterrows():
    adj[row[PARENT_COL]].append((row[COM_COL], float(row[QTY_COL])))
    
#trong trường hợp chỉ xài pure dictionary
    # adj = {}
    # if parent not in adj:
    #     adj[parent] = []
    # adj[parent].append(child)


"""1. the DFS algorithm"""

rows = []
for root in ROOTS:
    # A. Initialize stack (state vector): (parent, cumulative_qty, path, level)
    stack = [(root, 1.0, [root], 0)]
    # optional: Add root node itself as level 0
    if INCLUDE_ROOT:
        rows.append({
            'root': root,
            'parent_path': '',
            'child': root,
            'level': 0,
            'qty_per_parent': 1.0,
            'cumulative_qty': 1.0,
            'full_path': root
        })
    # B. While the task is still, keep exploring
    while stack:
        print("\n***CÔNG VIỆC***")
        print(stack)
        # B1. DFS behavior (LIFO, get the most recent task from the list) + "What is the current TASK?"
        parent, cum_qty, path, level = stack.pop() #pop from end
        print("Parent = Currently", parent)
        # B2. Child Processing (in the current parent)
        # Get all the child, down-one-level neighbors
        for child, rel_qty in adj.get(parent, []):
            print("CHILD: ",child, rel_qty)
            # optional: Cycle detecting
            if child in path:
                if STOP_ON_CYCLE:
                    raise ValueError(f"Cycle detected: {' > '.join(path + [child])}")
                continue

            new_cum = cum_qty * rel_qty
            new_path = path + [child]
            # Get info, record that child
            rows.append({
                'root': root,
                'parent_path': ' > '.join(path),
                'child': child,
                'level': level + 1,
                'qty_per_parent': rel_qty,
                'cumulative_qty': new_cum,
                'full_path': ' > '.join(new_path)
            })

            # Push deeper expansion , add task into the list
            stack.append((child, new_cum, new_path, level + 1))

exploded_df = pd.DataFrame(rows)
#exploded_df = exploded_df.sort_values(['root', 'level', 'child']).reset_index(drop=True)

exploded_df

### base - recursive way

In [ ]:
import pandas as pd
from collections import defaultdict

def explode_bom(flat_bom_df, roots, parent_col='parent', comp_col='component', qty_col='qty',
                include_root=True, stop_on_cycle=True):
    if isinstance(roots, (str, int)):
        roots = [roots]

    df = flat_bom_df.copy()
    if qty_col not in df.columns:
        df[qty_col] = 1.0

    # adjacency dict
    adj = defaultdict(list)
    for _, row in df.iterrows():
        adj[row[parent_col]].append((row[comp_col], float(row[qty_col])))

    rows = []

    def dfs(parent, cum_qty, path, level, root):
        for child, rel_qty in adj.get(parent, []):
            if child in path:
                if stop_on_cycle:
                    raise ValueError(f"Cycle detected: {' > '.join(path + [child])}")
                continue

            new_cum = cum_qty * rel_qty
            new_path = path + [child]

            # append here before going deeper (pre-order DFS)
            rows.append({
                'root': root,
                'parent_path': ' > '.join(path),
                'child': child,
                'level': level + 1,
                'qty_per_parent': rel_qty,
                'cumulative_qty': new_cum,
                'full_path': ' > '.join(new_path)
            })

            # recurse deeper
            dfs(child, new_cum, new_path, level + 1, root)

    for root in roots:
        if include_root:
            rows.append({
                'root': root,
                'parent_path': '',
                'child': root,
                'level': 0,
                'qty_per_parent': 1.0,
                'cumulative_qty': 1.0,
                'full_path': root
            })
        dfs(root, 1.0, [root], 0, root)

    exploded_df = pd.DataFrame(rows)
    return exploded_df


### --------- PRINTING RESULT
data = [
    ('A', 'B', 2),
    ('A', 'C', 1),
    ('B', 'D', 3),
    ('B', 'E', 2),
    ('C', 'F', 4),
    ('E', 'G', 1)
]
flat_bom = pd.DataFrame(data, columns=['parent', 'component', 'qty'])
exploded = explode_bom(flat_bom, roots=['A'], include_root=True)
print(exploded[['child','level']])

## 02. Bom level compute

- real name : item_level compute (from single lv bom)

### base

In [ ]:
# input data and parameter

import pandas as pd
from collections import defaultdict, deque
"""
A
    B
        D
        E
            G
    C
        F
"""
raw_data = [
    ('A', 'B', 2),
    ('A', 'C', 1),
    ('A', 'H', 100),
    ('B', 'D', 3),
    ('B', 'E', 2),
    ('B', 'G', 2),
    ('C', 'F', 4),
    ('C', 'G', 1)
]
df_bom = pd.DataFrame(raw_data, columns=['parent', 'component', 'qty'])

external_demand = pd.DataFrame([
    {"item": "A", "week": 4, "qty": 100},
])


def all_items(bom: pd.DataFrame, external: pd.DataFrame):
    items = set(external["item"].unique()) 
    items |= set(bom["parent"].unique()) #Union for Sets.
    items |= set(bom["component"].unique())
    return sorted(items)

def children_map(bom: pd.DataFrame):
    cm = defaultdict(list)
    for _, r in bom.iterrows():
        cm[r["parent"]].append((r["component"], r["qty"]))
    return cm

def compute_levels(bom: pd.DataFrame, roots):
    dict_bom = children_map(bom)
    level = {} # Your ouput - will store: {item: level_number}
    queue = deque() # Your task management (to run the BFS algorithm)
    for r in roots:
        level[r] = 0
        queue.append(r)
    while queue:
        p = queue.popleft()
        for c, _ in dict_bom.get(p, []):
            if c not in level or level[c] < level[p] + 1:
                level[c] = level[p] + 1
                queue.append(c)
    return level


In [ ]:
# run the algorithm
compute_levels(df_bom, ['A','B'])

### study (run with base)

In [ ]:
"""A. parameter"""

BOM = df_bom
roots = ['A']
dict_bom = children_map(BOM)

"""B. BFS algorithm"""

# B1. initialization

level = {} # Your ouput - will store: {item: level_number}
queue = deque() # Your task management (to run the BFS algorithm)

# B2. logic running
for r in roots:
    level[r] = 0 # Step 0: every input_item is set to level 0
    queue.append(r) # Step 1: Add to all item to queue for processing
    print("START LOOP: ", queue)
print("*"*10)
while queue:
    print(f"current {queue}")
    p = queue.popleft() # Step 2: Get next item from FRONT of queue (FIFO)
    print(f"START LOOP: , current processing {p}, queue is left {queue}, ")
    # Step 3: Start processing Child at the current 
    for c, _ in dict_bom.get(p, []): #Step 3.1 : Get the all child from that item
        
        #Step 3.2 Filter: A. First time seeing this child B. Seen before, but found at deeper level
        # condition B. Lúc search parent để tìm component dưới
            # Previously: Bolt was found at Level 2 , 
            # Now: Found the child component Bolt again via a Level 2 parent Assembly → Bolt would be Level 3
        
        if c not in level or level[c] < level[p] + 1: 
            #Step 3.3 If that happen , UPDATE that child to deeper level! then Re-process its children
            level[c] = level[p] + 1 
            queue.append(c)

"""PRINTING RESULT"""
print("\nRESULT IS {item : level}")
print(level)

print("\n ALL ITEM FOR MRP")
all_item = all_items(bom = BOM, external = external_demand)
print(all_item)
item_sort_by_level = sorted(all_item, key=lambda x: level.get(x, 10**6) ) #if not in dict , put it last
print(item_sort_by_level)


In [ ]:
"""
A
    G
    B
        D
        E
            G
    C
        F
"""

### phiên bản cải thiện (trả ra df như excel)

1. Add Cycle Detection
2. Add Build adjacency (children) map efficiently instead of iterativerow LOOP
3. Add Data validation

In [ ]:
raw_data = [
    ('A', 'B', 2),
    ('A', 'C', 1),
    ('A', 'G', 100),
    ('B', 'D', 3),
    ('B', 'E', 2),
    ('C', 'F', 4),
    ('E', 'G', 1)
]
df_bom = pd.DataFrame(raw_data, columns=['parent', 'component', 'qty'])

external_demand = pd.DataFrame([
    {"item": "A", "week": 4, "qty": 100},
])


In [ ]:
import pandas as pd
from collections import defaultdict, deque

def compute_bom_levels_bfs(bom: pd.DataFrame, roots: list[str]) -> pd.DataFrame:
    """
    Compute item levels in a multi-level BOM using Breadth-First Search (BFS).

    Args:
        bom : pd.DataFrame
            Must contain columns ['parent', 'component', 'qty'].
        roots : list[str]
            List of top-level parent items (e.g., finished goods).

    Returns:
        pd.DataFrame
            Columns:
            ['parent', 'component', 'qty', 'level_parent', 'level_component']
            where each item’s true level (depth) is computed across the BOM graph.
    """

    # ------------------------------------------------------------------
    # 1. Data validation
    # ------------------------------------------------------------------
    
    required_cols = {'parent', 'component', 'qty'}
    ## columns_name matching
    if not required_cols.issubset(bom.columns): 
        raise KeyError(f"BOM missing columns: {required_cols - set(bom.columns)}")
    ## ensure BOm has no null value
    if bom.isnull().any().any():
        raise ValueError("BOM contains null values")
    ## Deduplicate to ensure clean graph edges
    bom = bom.drop_duplicates(subset=['parent', 'component'])

    # ------------------------------------------------------------------
    # 2. Build adjacency (children) map efficiently
    # ------------------------------------------------------------------
    dict_bom = defaultdict(list)
    for product, child, qty in zip(bom['parent'], bom['component'], bom['qty']):
        dict_bom[product].append( (child, qty) )

    # ------------------------------------------------------------------
    # 3. Initialize BFS traversal
    # ------------------------------------------------------------------
    
    level = {} # Your ouput - will store: {item: level_number}
    queue = deque() # Your task management (to run the BFS algorithm)

    for r in roots:
        level[r] = 0
        queue.append(r)

    visited_edges = set()  # for cycle detection
    
    # ------------------------------------------------------------------
    # 4️⃣ BFS traversal
    # ------------------------------------------------------------------
    
    while queue:
        parent = queue.popleft()
        # ????parent_level = level[parent]
        
        # Iterate over all children of this parent
        for child, _ in dict_bom.get(parent, []):
            edge = (parent, child)
            
            # Cycle detection
            if edge in visited_edges: #gia su cap A,B va A,B o 2 level khac nhau thi sao ?
                pass  # do no skip the loop, because it affect the next computation
            if child == parent: #A → A
                raise ValueError(f"Self-cycle detected: {parent} → {child}")
            if (child, parent) in visited_edges: #(A → B) vs (B → A)
                raise ValueError(f"Cycle detected: {parent} ↔ {child}")
            
            visited_edges.add(edge)
            
            # Assign or update level
            
            if child not in level or level[child] < level[parent] + 1:
                level[child] = level[parent] + 1
                queue.append(child)


    # ------------------------------------------------------------------
    # 5️⃣ Prepare output (extensible for downstream use)
    # ------------------------------------------------------------------
    
    data = list( level.items() )
    df_level = pd.DataFrame(data, columns = ['item','level'])
    
    
    return df_level

In [ ]:
#compute_bom_levels_bfs(bom = df_bom, roots = ['A','B'])



# test on excel file
df_bom_excel = pd.read_excel("MRP_logic.xlsm",sheet_name= "input_bom")
df_bom_excel = df_bom_excel[['Parent','Child','Qty']]
df_bom_excel.columns = ['parent', 'component', 'qty']


df_item_excel = pd.read_excel("MRP_logic.xlsm",sheet_name= "input_item")
item_list = df_item_excel['ItemCode'].unique().tolist()


result = compute_bom_levels_bfs(bom = df_bom_excel, roots = item_list).sort_values(by ='level')
result

# function

## support/helper

In [ ]:
import math
import pandas as pd
from collections import defaultdict, deque

# -----------------------
# Helper / policy utils
# -----------------------
def round_up_to_multiple(q, multiple):
    if multiple <= 1:
        return int(q)
    return int(((q + multiple - 1) // multiple) * multiple)

def date_range_index(start_date, end_date):
    return pd.date_range(start=start_date, end=end_date, freq="D")


def compute_levels(bom, item_list):
    "create a list of item sorted based on bom's true level"
    
    parent_col, comp_col = ["parent", "component"]
    
    
    ## 1.1 Create dict bom
    dict_bom = defaultdict(list)
    for _, r in bom.iterrows():
        dict_bom[r[parent_col]].append(r[comp_col])

    ## 1.2 Find Bom level for each item
    level = {}
    queue = deque()
    for item in item_list:
        level[item] = 0
        queue.append(item)
        
    while queue:
        p = queue.popleft()
        for c in dict_bom.get(p, []):
            if c not in level or level[c] < level[p] + 1:
                level[c] = level.get(p, 0) + 1
                queue.append(c)
    # 2. Sorting based on bom LEVEL
    ordered = sorted(list(set(item_list)), key=lambda x: level.get(x, 0)) #parents/low lv first
    
    print("List of ALL ITEM LIST , sorted by true bom level:")
    print("------>: ",level)
    return ordered



def find_min_max_date(item_code: str, 
                      independent_demand_dict, 
                      global_dependent_dict, 
                      sr_dict):
    print(f"START find date range of {item_code}")
    
    # prepare
    independ_col, depend_col, sr_col = ['date', "req_date", "date"]
    
    def safe_df(source, key, default_col): 
        if key in source.keys() and source[key]: #"Only proceed if the key exists and its value is not empty or None."
            return pd.DataFrame(source[key])
        return pd.DataFrame(columns=[default_col])
    
    # create pd.df for each input
    
    df_ind = safe_df(independent_demand_dict, item_code, independ_col)
    df_dep = safe_df(global_dependent_dict, item_code, depend_col)
    df_sr = safe_df(sr_dict, item_code, sr_col)

    # filter non empty series for concat
    series_list = [
        df_ind.get(independ_col, pd.Series(dtype='datetime64[ns]')),
        df_dep.get(depend_col, pd.Series(dtype='datetime64[ns]')),
        df_sr.get(sr_col, pd.Series(dtype='datetime64[ns]'))
    ]
    non_empty_series = [s for s in series_list if not s.empty]
    
    
    if non_empty_series:
        all_dates = pd.concat(non_empty_series, ignore_index=True)
    else:
        all_dates = pd.Series(dtype='datetime64[ns]')
    # validity check
    
    if all_dates.empty or pd.isna(all_dates.min()) or pd.isna(all_dates.max()):
        print(f"-->[WARN] Item {item_code} has no valid date values.")
        return None, None
        
    # ---- get values -----
    
    start_date = all_dates.min()
    end_date = all_dates.max()
    
    print(f"-->[INFO] {item_code} | Start: {start_date} | End: {end_date}")
    
    
    
    
    
    return start_date, end_date



def prepare_input(list_of_df):
    
    
    demand_df = list_of_df['transaction']['demand_orders']
    onhand_df = list_of_df['transaction']['onhand']
    supply_df = list_of_df['transaction']['supply_orders']
    bom_df = list_of_df['master']['bom_master']
    item_df = list_of_df['master']['item_master']
    policy_df = list_of_df['master']['policy_master']

    
    #------------------------------------
    items = sorted(set(bom_df["parent"])
               .union(set(bom_df["component"]))
               .union(set(demand_df["item"])) 
               )
    items_ordered = compute_levels(bom = bom_df, item_list= items)
    
    #------------------------------------
    demand_by_item = defaultdict(list)
    sr_by_item = defaultdict(list)
    
    for _, row in demand_df.iterrows() :
        demand_by_item[row["item"]].append({
            "item": row["item"],
            "date": row["date"],
            "qty": float(row["qty"])
        })
    for _, row in supply_df.iterrows() :
        sr_by_item[row["item"]].append({
            "item": row["item"],
            "date": row["date"],
            "qty": float(row["qty"])
    })
    #------------------------------------
    
    onhand_dict = onhand_df.set_index("item_code").to_dict(orient="index")
    item_master_dict = item_df.set_index("item_code").to_dict(orient="index")
    policy_dict = policy_df.set_index("item_code").to_dict(orient="index")
    
    data = {
        "item_ordered": items_ordered,
        "demand_by_item": demand_by_item,
        "sr_by_item": sr_by_item,
        "onhand_dict": onhand_dict,
        "item_master_dict": item_master_dict,
        "policy_dict": policy_dict,
        "bom_df": bom_df,

    }
    
    return data



import numpy as np



def safe_policy(self, item, policy_dict, default_policy, valid_values):
    merged = {**default_policy, **policy_dict.get(item, {})}
    cleaned = {}
    
    def is_missing(v): 
        """Return True if the value is considered empty / invalid."""
        return (
            v is None
            or (isinstance(v, float) and math.isnan(v))
            or (isinstance(v, str) and v.strip() == "")
            or pd.isna(v)
        )
    
    
    for k, v in merged.items():
        #Handle all case: None, NaN, "nan", "", etc.
        if is_missing(v):
            cleaned[k] = default_policy[k]
            continue
    
        # String case: normalization & validation
        if isinstance(v, str):
            v_clean = v.strip().upper()
            if k in valid_values:
                if v_clean in valid_values[k]:
                    cleaned[k] = v_clean.title() if k == "week_day" else v_clean
                else:
                    cleaned[k] = default_policy[k] #value not in valid ----> just use default
            else:
                cleaned[k] = v #key not in valid ----> maybe numeric col
        else:
            cleaned[k] = v #value not string --> maybe numeric col
            
    
    policy_name = cleaned['policy_name']
    
    key_cv, key_wd, key_minmax = ["cover_days", "week_day", "max_level"]
    
    if policy_name == "COVER_DAYS":
        cleaned[key_wd] = np.nan
        cleaned[key_minmax] = np.nan
    elif policy_name == "MIN_MAX":
        cleaned[key_cv] = np.nan
        cleaned[key_wd] = np.nan
    elif policy_name == "WEEKLY_CALENDAR":
        cleaned[key_minmax] = np.nan
        cleaned[key_cv] = np.nan
    else:
        cleaned[key_cv] = np.nan
        cleaned[key_wd] = np.nan
        cleaned[key_minmax] = np.nan
        
    
    return cleaned





import os
import pandas as pd
from datetime import datetime


base_output_dir = r"./data/output"


def writting_result(data: dict):
    # Create timestamp folder
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    output_dir = os.path.join(base_output_dir, timestamp)
    os.makedirs(output_dir, exist_ok=True) #create
    print(f"START WRITING RESULT AT FOLDER {base_output_dir}")
    
    # Create custom func for writing
    def _write_dict(d, prefix=""):
        for key, value in d.items():
            if isinstance(value, pd.DataFrame):
                filename = f"{prefix}{key}.csv" if not prefix else f"{prefix}_{key}.csv"
                filepath = os.path.join(output_dir, filename)
                value.to_csv(filepath, index=True)
                print(filepath)
            elif isinstance(value, dict):
                _write_dict(value, prefix=f"{prefix}{key}" if not prefix else f"{prefix}_{key}")
            else:
                print(f"[SKIP] {prefix}{key}: Not a DataFrame or dict")
                
    # DO WRITING  
    _write_dict(data)

## MRP engine

In [ ]:
def item_df_initializing(item_code, cal_cols, date_index, 
                            on_hand: float, 
                            independent_demand_dict, global_dependent_dict, sr_dict):
    """
    - INPUT:
        - KHUNG: item_code, columns_list, item_index, 
        - VALUE: independent_demand_dict, global_dependent_dict, sr_dict
        - VALUE: on-hand
    - TASK:
        - 1. Tạo khung table: columns + index
        - 2. Seeding day0 inventory
        - 3. Seeding Gross demand (independent + dependent)
        - 4. Seeding Schedule receipt (supply_order)

    """
    
    gross_col = cal_cols[0] # 'Gross Req'
    sr_col = cal_cols[1] #'Scheduled Rcpt'
    poh1_col, poh2_col = cal_cols[2], cal_cols[5] #'POH 1 (before LS)' , 'POH 2 (after LS)'
    
    result_df = pd.DataFrame(0, index = date_index, columns= cal_cols)
    day0 = date_index[0]
    result_df.at[day0, poh1_col] = on_hand
    result_df.at[day0, poh2_col] = on_hand
    
    
    
    
    ## seeding GROSS INDENPEDENT DEMAND
    for rec in independent_demand_dict.get(item_code, []):
        d = rec["date"]
        qty = rec["qty"]
        if d in result_df.index:
            result_df.at[d, gross_col] += int(qty)
    ## seeding GROSS DEPENDENT DEMAND
    for rec in global_dependent_dict.get(item_code, []):
        d = rec["req_date"]
        qty = rec["gross_req_qty"]
        if d in result_df.index:
            result_df.at[d, gross_col] += int(qty)
    ## seeding SCHEDULE RECEIPT
    for rec in sr_dict.get(item_code, []):
        d = rec["date"]
        qty = rec["qty"]
        if d in result_df.index:
            result_df.at[d, sr_col] += int(qty)

    return result_df


def calculate_net_requirement(tp_item_df, cal_cols: list = None):
    """
    Return a df of result
    - only net requirement before lot size + before safety stock
    - tp_item_df indexed by datetime.date or pd.Timestamp
    - cal_cols: list of named column
    """
    
    gross_col = cal_cols[0]
    sched_col = cal_cols[1]
    poh1_col = cal_cols[2] #before LS
    net1_col = cal_cols[3] #before LS
    
    
    for curr_date in tp_item_df.index:
        # 1. GET CURRENT VALUE
        gross = int(tp_item_df.at[curr_date, gross_col]) if curr_date in tp_item_df.index else 0
        sched = int(tp_item_df.at[curr_date, sched_col]) if curr_date in tp_item_df.index else 0
        
        
        # HANDLING previous_date index and previous day on-hand
        first_day = tp_item_df.index.min()
        if curr_date == first_day:
            onhand_prev = int(tp_item_df.at[first_day, poh1_col])
        else:
            prev = curr_date - pd.Timedelta(days=1)
            # If prev not in index (shouldn't happen), fallback to first day
            if prev not in tp_item_df.index:
                prev = first_day
    
            onhand_prev = int(tp_item_df.at[prev, poh1_col])
        #------------------
        
        # Calculating net value
        available = onhand_prev + sched #-safety_stock
        usable = max(0, available)
        
        
        
        net = 0 if usable >= gross else gross - usable

        # 2. FILL / UPDATE
        # Net Requirement
        tp_item_df.at[curr_date, net1_col] = int(net) if curr_date != first_day else 0 #set up day 0 is 0

        # POH (skip if net < 0)
        tp_item_df.at[curr_date, poh1_col] = int(max(0, usable - gross)) if curr_date != first_day else onhand_prev #set up day 0 is 0


def calculate_lot_size(tp_item_df, safety_stock=0, cal_cols: list = None,
                       policy_name = "L4L", policy_param: dict = None):
    #GET columns
    gross_col = cal_cols[0]
    sched_col = cal_cols[1]
    poh1_col = cal_cols[2] #before LS
    net1_col = cal_cols[3] #before LS
    
    ls_col = cal_cols[4] #lot size decision
    poh2_col = cal_cols[5] #after LS
    net2_col = cal_cols[6] #after LS

    
    
    
    #L4L + FIXED CASE 
    if policy_name in ["L4L","FIXED"]:
        
        for curr_date in tp_item_df.index[1:]:
            # 0. GET value for calc
            lot_size_decision_qty = 0
            gross = int(tp_item_df.at[curr_date, gross_col])
            sched = int(tp_item_df.at[curr_date, sched_col])
            onhand_prev = int(tp_item_df.at[curr_date - pd.Timedelta(days=1), poh2_col])
            
            # 1. UPDATE VALUE + FILL ROW
            inventory_position = onhand_prev + sched - gross + lot_size_decision_qty
            poh2_qty = max(inventory_position,0)
            
            tp_item_df.at[curr_date, poh2_col] = poh2_qty
            tp_item_df.at[curr_date, net2_col] = inventory_position
            
            # 2. Do it 2nd time only when NET OCCUR
            if inventory_position < safety_stock:
                # no safety stock then inv_pos < 0 is OK
                # yes safety stock then below safetystock is OK --> split 2 TH net2_qty nega or positive
                # COMPUTE Lot size
                lot_size_decision_qty = abs(inventory_position) + safety_stock if inventory_position <=0 else abs(inventory_position - safety_stock)
                
                """handle MOQ and Rouding value"""
                
                moq = policy_param.get("MOQ",1)
                lot_size_decision_qty = max(lot_size_decision_qty,moq)
                
                rounding_value = policy_param.get("rounding_value",1)
                lot_size_decision_qty = round_up_to_multiple(lot_size_decision_qty, rounding_value)
                """----------------------------------------"""
                # UPDATE inventory_position , poh2,net2
                inventory_position = onhand_prev + sched - gross + lot_size_decision_qty
                poh2_qty = max(inventory_position,0) #do not minus
                
                # FILL ROW
                tp_item_df.at[curr_date, ls_col] = lot_size_decision_qty
                tp_item_df.at[curr_date, poh2_col] = poh2_qty
                tp_item_df.at[curr_date, net2_col] = inventory_position

    elif policy_name == "COVER_DAYS" :
        
        last_index = tp_item_df.index.max() #xài index[-1] nếu đã sort
        n_cover_days = policy_param.get("cover_days", 7)
        
        
        for curr_date in tp_item_df.index[1:]:
            # 0. GET value for calc
            lot_size_decision_qty = 0
            gross = int(tp_item_df.at[curr_date, gross_col])
            sched = int(tp_item_df.at[curr_date, sched_col])
            onhand_prev = int(tp_item_df.at[curr_date - pd.Timedelta(days=1), poh2_col])
            # 1. UPDATE VALUE + FILL ROW
            
            inventory_position = onhand_prev + sched - gross + lot_size_decision_qty
            poh2_qty = max(inventory_position,0)
            
            tp_item_df.at[curr_date, poh2_col] = poh2_qty
            tp_item_df.at[curr_date, net2_col] = inventory_position
            
            # 2. Do it 2nd time only when NET OCCUR
            if inventory_position < 0:
                end_calc_date = min(last_index, curr_date + pd.Timedelta(days=n_cover_days))
                
                lot_size_decision_qty = 0
                for curr_calc_date in date_range_index(curr_date, end_calc_date):
                    fake_lot_size = 0
                    gross2 = int(tp_item_df.at[curr_calc_date, gross_col])
                    sched2 = int(tp_item_df.at[curr_calc_date, sched_col])
                    onhand_prev2 = int(tp_item_df.at[curr_calc_date - pd.Timedelta(days=1), poh2_col])
                    
                    inventory_position2 = onhand_prev2 + sched2 - gross2 + fake_lot_size
                    poh2_qty2 = max(inventory_position2,0)
                    
                    tp_item_df.at[curr_calc_date, poh2_col] = poh2_qty2
                    tp_item_df.at[curr_calc_date, net2_col] = inventory_position2
                    
                    
                    if inventory_position2 < 0:
                        lot_size_decision_qty += abs(inventory_position2)
                        
                """safety stock handling"""
                
                lot_size_decision_qty = lot_size_decision_qty + safety_stock
                """handle MOQ and Rouding value"""
                
                moq = policy_param.get("MOQ",1)
                lot_size_decision_qty = max(lot_size_decision_qty,moq)
                
                rounding_value = policy_param.get("rounding_value",1)
                lot_size_decision_qty = round_up_to_multiple(lot_size_decision_qty, rounding_value)
                """---------------------------------"""
                # UPDATE inventory_position , poh2,net2

                inventory_position = onhand_prev + sched - gross + lot_size_decision_qty
                
                poh2_qty = max(inventory_position,0) #do not negative
                
                # FILL ROW
                tp_item_df.at[curr_date, ls_col] = lot_size_decision_qty
                tp_item_df.at[curr_date, poh2_col] = poh2_qty
                tp_item_df.at[curr_date, net2_col] = inventory_position
                
    elif policy_name == "WEEKLY_CALENDAR":
        weekday_list = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
        weekday_parameter = policy_param.get("week_day","Monday")
        weekday_order_num = weekday_list.index(weekday_parameter)
        #print(weekday_order_num)
        last_index = tp_item_df.index.max()
        
        curr_date = tp_item_df.index[1] #get day 1 - today
        
        while curr_date < last_index: #k duoc de dau bang, neu khong se infinite loop
            
            # 0. finding next review (tuesday or last_index)
            next_review_raw = curr_date + pd.offsets.Week(weekday=weekday_order_num, n = 1)
            next_review_true = min(next_review_raw, last_index)
            #print(f"current {curr_date} Raw next {next_review_raw}, True next {next_review_true}")

            lot_size_decision_qty = 0
            # 1. MAIN - LOOP dateX/thứ 3 tuần này cover tới t2 tuần sau
            for curr_calc_date in date_range_index(curr_date, next_review_true - pd.Timedelta(days=1)):
                # 0. GET value for calc
                fake_lot_size = 0
                gross = int(tp_item_df.at[curr_calc_date, gross_col])
                sched = int(tp_item_df.at[curr_calc_date, sched_col])
                onhand_prev = int(tp_item_df.at[curr_calc_date - pd.Timedelta(days=1), poh2_col])
                
                # 1. UPDATE VALUE + FILL ROW
                inventory_position = onhand_prev + sched - gross + fake_lot_size
                poh2_qty = max(inventory_position,0)
                
                tp_item_df.at[curr_calc_date, poh2_col] = poh2_qty
                tp_item_df.at[curr_calc_date, net2_col] = inventory_position
                
                lot_size_decision_qty += abs(inventory_position) if inventory_position < 0 else 0
                #print(f"Lot size of {curr_date} is {lot_size_decision_qty}")
                
                
            # 1.5 MAIN - IF lot size > 0, recalculation the LOOP window
            if lot_size_decision_qty > 0:
                
                """safety stock handling"""
                
                lot_size_decision_qty = lot_size_decision_qty + safety_stock
                """handle MOQ and Rouding value"""
                
                moq = policy_param.get("MOQ",1)
                lot_size_decision_qty = max(lot_size_decision_qty,moq)
                
                rounding_value = policy_param.get("rounding_value",1)
                lot_size_decision_qty = round_up_to_multiple(lot_size_decision_qty, rounding_value)
                """---------------------------------"""
                
                for curr_calc_date in date_range_index(curr_date, next_review_true - pd.Timedelta(days=1)):
                    gross = int(tp_item_df.at[curr_calc_date, gross_col])
                    sched = int(tp_item_df.at[curr_calc_date, sched_col])
                    onhand_prev = int(tp_item_df.at[curr_calc_date - pd.Timedelta(days=1), poh2_col])
                    # 1. UPDATE VALUE + FILL ROW, LOT SIZE DECISION AT DAY 1 of the loop
                    if curr_calc_date == curr_date:
                        inventory_position = onhand_prev + sched - gross + lot_size_decision_qty
                    else:
                        inventory_position = onhand_prev + sched - gross + 0
                    poh2_qty = max(inventory_position,0)

                    
                    tp_item_df.at[curr_calc_date, ls_col] = lot_size_decision_qty
                    tp_item_df.at[curr_calc_date, poh2_col] = poh2_qty
                    tp_item_df.at[curr_calc_date, net2_col] = inventory_position
                    
                    lot_size_decision_qty = 0 #reset to next day UPDATING
            
            # SKIP TO NEXT REVIEW
                # using this if có dấu bang
                # if next_review_true == curr_date:
                #     break
            
            curr_date = next_review_true
            
    elif policy_name == "MIN_MAX":
        
        s = safety_stock
        S = policy_param.get("max_level", 100000)
        
        for curr_date in tp_item_df.index[1:]:
            # 0. GET value for calc
            lot_size_decision_qty = 0
            gross = int(tp_item_df.at[curr_date, gross_col])
            sched = int(tp_item_df.at[curr_date, sched_col])
            onhand_prev = int(tp_item_df.at[curr_date - pd.Timedelta(days=1), poh2_col])
            
            # 1. UPDATE VALUE + FILL ROW
            inventory_position = onhand_prev + sched - gross + lot_size_decision_qty
            poh2_qty = max(inventory_position,0)
            
            tp_item_df.at[curr_date, poh2_col] = poh2_qty
            tp_item_df.at[curr_date, net2_col] = inventory_position
            
            # 2. Do it 2nd time only when NET OCCUR
            if inventory_position < s and s < S:
                lot_size_decision_qty = abs(S - inventory_position)
                
                """handle MOQ and Rouding value"""
                
                moq = policy_param.get("MOQ",1)
                lot_size_decision_qty = max(lot_size_decision_qty,moq)
                
                rounding_value = policy_param.get("rounding_value",1)
                lot_size_decision_qty = round_up_to_multiple(lot_size_decision_qty, rounding_value)
                """----------------------------------------"""
                # UPDATE inventory_position , poh2,net2
                inventory_position = onhand_prev + sched - gross + lot_size_decision_qty
                poh2_qty = max(inventory_position,0) #do not minus
                
                # FILL ROW
                tp_item_df.at[curr_date, ls_col] = lot_size_decision_qty
                tp_item_df.at[curr_date, poh2_col] = poh2_qty
                tp_item_df.at[curr_date, net2_col] = inventory_position
                
                
                #ENSURE s < S
                if s >= S:
                    raise Exception(f"Your s {s} is greater than S {S} at XXX-item(chưa tìm cách bỏ vô đc).")
                    
def calculate_planned_order(
    item_name: str, tp_item_df: "pd.DataFrame", 
    procurement_type: str, lead_time: int, 
    today , output_cols: list = None) -> "pd.DataFrame":
    """
    - What: Order recommendation for the item
    - Task: Calculates planned order quantities. (Prod Ord / Pur Ord)
    - Inputs:
        - tp_item_df: DataFrame with input column [date, lot_size_decision_qty]
        - lead_time: 
        - procurement_type: to define purchase order / production order
        - today: date-like for 'URGENT' flag (defaults to local today)
    - Returns:
        - DataFrame with columns: ["item", "planned_qty", "receipt_date", "release_date", "lead_time_days" ,"type" , "urgency"]
    """
    # --- 0. OUTPUT INITILIZATION ---
    
    ## COLS_PLANNED_ORDER = ["item", "planned_qty", "receipt_date", "release_date" ,"type" , "urgency_status", "lead_time"]
    item_col = output_cols[0]
    qty_col = output_cols[1]
    receipt_date_col = output_cols[2]
    release_date_col = output_cols[3] 
    type_col = output_cols[4] 
    urgency_col = output_cols[5]
    lead_time_cols = output_cols[6]
    ## OUTPUT
    output_df = tp_item_df.copy()
    
    
    # --- 0. COMPUTE LOGIC ---
    
    output_df[item_col] = item_name
    output_df[receipt_date_col] = output_df.index #or df.reset_index().rename(columns={'index': 'timestamp'})
    output_df[release_date_col] = output_df[receipt_date_col] - pd.to_timedelta(lead_time, unit="D")
    output_df[lead_time_cols] = lead_time
    output_df.rename(columns={'Lot size': qty_col}, inplace = True)
    
    type_mapping = {
    "FNG": "Production Order",
    "ASSEMBLY": "Production Order",
    "RAW" : "Purchase Order"
    }
    output_df[type_col] = type_mapping.get(procurement_type, "Purchase Order")
    
    
    
    # CÁCH 2 - NHƯ CẶC, QUÁ NẶNG
    # if procurement_type in ["FNG","ASSEMBLY"]:
    #     output_df[type_col] = "Production Order"
    # else:
    #     output_df[type_col] = "Purchase Order"
    # CÁCH 3: VECTORIZE, CHẠY C speed
    # output_df["type"] = np.where(
    # output_df["procurement_type"].isin(["FNG", "ASSEMBLY"]),
    # "Production Order",
    # "Purchase Order"
    output_df[urgency_col] = ""
    output_df.loc[output_df[release_date_col] <= today , urgency_col] = "urgent"
    
    
    # REORDER COLUMNS
    desired_order = [item_col, qty_col] + [col for col in output_df.columns if col not in [item_col, qty_col]]
    output_df = output_df[desired_order]

    return output_df.reset_index(drop = True)


def exploding_parent_item(
    item_name: str, lead_time: int,
    bom_df : "pd.DataFrame", 
    item_planned_order_df: "pd.DataFrame", scheduled_receipt = None, 
    input_cols: list = None, output_cols: list = None,
    ) -> "pd.DataFrame":
    """
    run only when production orders (assembly/FNG)
    - What: Exploding child's gross demand from (PARENT , non-RAW) item
    - Task: Calculating all child's qty from parent's order 
    - Inputs: planned_order dataframe following column ["item", "planned_qty", "receipt_date", "release_date", "order_type" , "urgency_status","lead_time"]
    - Returns:
        - Child-Gross Requirement DataFrame with columns: ["item", "gross_req_qty", "req_date","source", "source_item","urgency_status"]
    """
    # 0. Get columns variables
    
    item_input_col , planned_qty_col, receipt_date_col, release_date_col, order_type_col, urgency_input_col, lt_col = input_cols
    item_output_col, req_qty_col, req_date_col, source_col, source_item_col, urgency_output_col = output_cols
    
    


    # 1.1. Build adjacency map for fast lookup
    
    dict_bom = defaultdict(list)
    
    for product, child, qty in zip(bom_df['parent'], bom_df['component'], bom_df['qty_per']):
        dict_bom[product].append( (child, qty ))
    # BAD PERFORMANCE VERSION
    # for _, row in bom_df.iterrows():
    #     dict_bom[row['parent']].append( 
    #                               (row['component'], row['qty_per']) 
    #                             )
    
    
    
    # 1.2 Transform schedule receipt --> schedule release from dict(list) into pd.df
    
    tempt_records = [row for rec in scheduled_receipt.values() for row in rec]
    scheduled_order_df = pd.DataFrame(tempt_records)
    
    scheduled_order_df[item_input_col] = scheduled_order_df["item"].astype(str)
    scheduled_order_df[planned_qty_col] = scheduled_order_df["qty"].astype(float)
    
    scheduled_order_df[receipt_date_col] = scheduled_order_df["date"]
    scheduled_order_df[release_date_col] = scheduled_order_df[receipt_date_col] - pd.to_timedelta(lead_time, unit='d')
    scheduled_order_df[order_type_col] = ""
    scheduled_order_df[urgency_input_col] = ""
    scheduled_order_df[lt_col] = lead_time
    
    
    
    # 1.3 Append Planned Order + Schedule Release
    sources = {
    "planned_order": item_planned_order_df,
    "scheduled_receipt": scheduled_order_df
    }
    
    df_all = pd.concat(
        [df.assign( source =name) for name, df in sources.items() ]
        
    )

    
    # 2. Row-by-row computation for PLANNED ORDER + SCHEDULED_RELEASE
    
    exploded_dict = defaultdict(list)
    exploded_list = []

    for _, row in df_all.iterrows():
        parent_item = row[item_input_col]
        parent_qty = row[planned_qty_col]
        release_date = row[release_date_col]
        urgency = row[urgency_input_col]
        source = row[source_col]
        
        # each Row item: find ALL childs in BOM ---> writing result
        for child_item, child_qty_per_parent in dict_bom.get(parent_item, []):
            
            records = {
                item_output_col: child_item,
                req_date_col: release_date,
                req_qty_col: parent_qty * child_qty_per_parent,
                source_col: source,
                source_item_col:parent_item,
                urgency_output_col: urgency
            }
            
            exploded_list.append(records)
            exploded_dict[child_item].append(records)
            


    exploded_df = pd.DataFrame(exploded_list)
    
    return exploded_dict, exploded_df



## MRP display

In [ ]:
def step1_add_planned_col(input_mrp_df, input_planned_df):
    
    # ---------- reindex the main dataframe to include all dates ----------

    all_dates = (input_mrp_df.index
             .union(input_planned_df['receipt_date'])
             .union(input_planned_df['release_date'])
            ).sort_values()
    
    
    result_df = input_mrp_df.reindex(
    pd.date_range( all_dates.min(), all_dates.max(), freq='D' )
    )
    
    # --- aggregate planned qty by release/receipt date then turn into dict ---
    
    release_map = (
        input_planned_df.groupby("release_date")["planned_qty"]
        .sum().to_dict()
    )
    receipt_map = (
        input_planned_df.groupby("receipt_date")["planned_qty"]
        .sum().to_dict()
    )
    
    # --- map value of the columns to index ---


    result_df['planned_receipt'] = 0
    result_df['planned_release'] = 0

    result_df['planned_receipt'] = result_df.index.to_series().map(receipt_map)
    result_df['planned_release'] = result_df.index.to_series().map(release_map)
    # --- fill na value and sorting ---
    result_df = result_df.fillna(0).sort_index().astype(int)
    
    
    return result_df

def step2_add_balance_col(input_mrp_df, on_hand: float, bal_col, sum_col):
    #
    col_raw_bal, col_sr_bal, col_sr_pr_bal, col_sr_1pr_bal = bal_col
    col_gross , col_sr, col_p_reiv, col_p_rls= sum_col
    
    
    
    # add day 0 inventory at 4 balance columns
    input_mrp_df.loc[input_mrp_df.index[0], bal_col] = on_hand
    
    # Identify first planned receipt date (if any)
    
    first_pr_date = (
    input_mrp_df.loc[input_mrp_df[col_p_reiv] > 0].index.min()
    if (input_mrp_df[col_p_reiv] > 0).any()
    else None
    )
    
    # Iterate from 2nd row onward to fill value
    
    for i in range(1, len(input_mrp_df)):
        
        # Define row and index
        prev_row = input_mrp_df.iloc[i - 1]
        curr_row = input_mrp_df.iloc[i]
        curr_idx = input_mrp_df.index[i]
        # Find component value
        gross = curr_row[col_gross]
        sched = curr_row[col_sr]
        plan_r = curr_row[col_p_reiv]
        
        add_first_pr = plan_r if curr_idx == first_pr_date else 0
        # Compute balance
        input_mrp_df.at[curr_idx, col_raw_bal] = prev_row[col_raw_bal] - gross
        input_mrp_df.at[curr_idx, col_sr_bal] = prev_row[col_sr_bal] + sched - gross
        input_mrp_df.at[curr_idx, col_sr_pr_bal] = prev_row[col_sr_pr_bal] + sched + plan_r - gross
        input_mrp_df.at[curr_idx, col_sr_1pr_bal] = prev_row[col_sr_1pr_bal] + sched + add_first_pr - gross
    
    return input_mrp_df

def step3_MRP_display(input_mrp_df, bal_col, sum_col, today, mode = "Daily"):
    
    if mode not in ['Daily', 'Weekly']:
        print("Your input MODE_HORIZON IS WRONG !!! CHOOSE AGAIN")
        return None

    # ------------
    agg_dict = {**{c: 'sum' for c in sum_col},
                **{c: 'last' for c in bal_col}}

    
    ## ------------ HANDLE BEFORE ------------
    df_before = input_mrp_df[input_mrp_df.index < today].copy()
    df_before['final_index'] = 'overdue'

    df_before = (
        df_before
        .groupby('final_index', as_index=True)
        .agg(agg_dict)
    )
    df_before.index.name = None
    
    ## ------------ HANDLE AFTER ------------
    
    if mode == "Daily":
        df_after = input_mrp_df[input_mrp_df.index >= today].copy()
        
        iso = df_after.index.isocalendar()

        df_after['final_index'] = [
            # Condition 1,2 : Monday or first index (even if not Monday) → add week marker
            f"{d.strftime('%d/%m')} ,, W{w:02d}Y{y % 100}" if d.weekday() == 0 or d == df_after.index[0]
            else f"{d.strftime('%d/%m')} ,, W{w:02d}Y{y % 100}" if d == df_after.index[0]
            # Condition 3: other days → plain date
            else d.strftime('%d/%m')
            for d, w, y in zip(df_after.index, iso.week, iso.year) #BẢN CHẤT VẪN LÀ TẠO COLUMNS
        ]
        df_after['final_index'] = df_after['final_index'].astype(str) + ","
        df_after = df_after.set_index('final_index')
        df_after.index.name = None

    elif mode == "Weekly": 
        df_after = input_mrp_df[input_mrp_df.index >= today].copy()
        df_after['week_number'] = df_after.index.to_series().dt.isocalendar().week
        df_after['year'] = df_after.index.to_series().dt.year % 100
        df_after['week_label'] = (
            "W" + df_after['week_number'].astype(int).astype(str).str.zfill(2)
            + "Y" + df_after['year'].astype(int).astype(str)
        )

        df_after = (
            df_after
            .groupby('week_label', as_index=True)
            .agg(agg_dict)
        )
        df_after.index.name = None
    
    else:
        return None
    
    ## APPENDING

    result_df = pd.concat([df_before, df_after])
    result_df = result_df.T
    result_df = result_df.reset_index().rename(columns={'index': 'MRP_element'})
    
    return result_df
    
def step4_add_info(input_mrp_df, item_code, item_desc, on_hand, item_policy_param):
    # INPUT
    lt = item_policy_param.get("lead_time", 1)
    ss = item_policy_param.get("safety_stock", 0)
    policy_name = item_policy_param.get("policy_name", "L4L") 
    rounding = item_policy_param.get("rounding_value", 1)
    moq = item_policy_param.get("MOQ", 1)
    
    cover_days = item_policy_param.get("cover_days", None)
    week_day = item_policy_param.get("week_day", None)
    max_level = item_policy_param.get("max_level", None)
    
    # DICT
    info_dict = {'Item Code': item_code,
             'Description': item_desc,
             'Current stock': on_hand,
             'Lead Time': lt,
             'Safety Stock': ss,
             'Order Policy': policy_name,
             'Rounding Value / MOQ': f"{rounding}/ {moq}",
             'Cover Days / Week Day / Max Level': f"{cover_days}/ {week_day}/ {max_level}"
    }
    
    # Convert dict → DataFrame

    info_df = pd.DataFrame(list(info_dict.items()), columns=['Field', 'Field Value'])
        
    # REINDEX
    if len(info_df) != len(input_mrp_df):
        print(f"Error: Unmatching columns between mrp and info")
        return

        # max_len = max(len(info_df), len(input_mrp_df))
        # info_df = info_df.reindex(range(max_len)).fillna('')
        # input_mrp_df = input_mrp_df.reindex(range(max_len)).fillna('')

    input_mrp_df = pd.concat([info_df, input_mrp_df], axis=1)

    # input_mrp_df['item_code_index'] = item_code
    # input_mrp_df = input_mrp_df.set_index('item_code_index')
    # final_mrp_df_by_item.index.name = None
    
    return input_mrp_df



# MRP script

## prepare source_data, computed_input and computed_ouput

In [ ]:

import math
import pandas as pd
from collections import defaultdict, deque

# 01. Nạp SOURCE
demand_order_df = pd.read_csv(r"data\transaction\demand_orders.csv", parse_dates=["date"])
onhand_df = pd.read_csv(r"data\transaction\onhand.csv", parse_dates=["date"])
supply_order_df = pd.read_csv(r"data\transaction\supply_orders.csv", parse_dates=["date"])



bom_df = pd.read_csv(r"data\master\bom_master.csv")
item_master_df = pd.read_csv(r"data\master\item_master.csv")
policy_master_df = pd.read_csv(r"data\master\policy_master.csv")




In [ ]:
# 02. Chuẩn bị ĐẦU VÀO và ĐẦU RA

items = sorted(set(bom_df["parent"])
               .union(set(bom_df["component"]))
               .union(set(demand_order_df["item"])) 
               )


# PRECOMPUTE variables before the MRP run (CHUAN BI DATA EFFICIENTLY TRUOC KHI BO VO LOOP)


items_ordered = compute_levels(bom = bom_df, item_list= items)
demand_by_item = defaultdict(list)
sr_by_item = defaultdict(list)

for _, row in demand_order_df.iterrows() :
    demand_by_item[row["item"]].append({
        "item": row["item"],
        "date": row["date"],
        "qty": float(row["qty"])
    })
for _, row in supply_order_df.iterrows() :
    sr_by_item[row["item"]].append({
        "item": row["item"],
        "date": row["date"],
        "qty": float(row["qty"])
    })



# GLOBAL VARIABLES
BOM_DF = bom_df.copy()
ONHAND_DICT = onhand_df.set_index("item_code").to_dict(orient="index")
ITEM_MASTER_DICT = item_master_df.set_index("item_code").to_dict(orient="index")
POLICY_DICT = policy_master_df.set_index("item_code").to_dict(orient="index")
ITEMS_ORDERED = items_ordered.copy()
TODAY = pd.Timestamp("2025-10-01")


COLS_MRP_COMPUTE = ["Gross Req", "Scheduled Rcpt", "POH 1 (before LS)", "Net Req 1 (before LS)", 
                    "Lot size","POH 2 (after LS)","Net Req 2 (after LS)" ]
COLS_PLANNED_ORDER = ["item", "planned_qty", "receipt_date", "release_date", "order_type" , "urgency_status","lead_time"]
COLS_GROSS_REQUIREMENT = ["item", "gross_req_qty", "req_date","source", "source_item","urgency_status"]

# ------------------------------------

# OUTPUT INITIALIZATION

planned_order_list = []
dependent_demand_list = []
global_dependent_demand_dict = defaultdict(list)
TP_MAP = {} #this is your ouput ,có copy hay không đều k bị ah , vì output dạng dictionary --> cả 2 đều hướng tới 1 table df như nhau
"""TASK HERE: """

# ----> so sán câu trúc full pd.df vs {item : pd.df}
# ----> về mặt slicing (rất ngu) --> pre-group (HỎI: có nhanh = dict không ???)

In [ ]:
demand_by_item

## mrp computation

In [ ]:

# MRP runnning
for item in ITEMS_ORDERED:
    #STEP 0 : GET THE PARAMETER
    print(f"----------------------------- START LOOP MRP of {item} ----------------------------------------------------------")
    on_hand = float( ONHAND_DICT[item].get("qty", 0) )
    policy_name = POLICY_DICT[item].get("policy_name", "L4L")
    policy_param = POLICY_DICT[item]
    ss_qty = POLICY_DICT[item].get("safety_stock", 0) #----- BỎ
    lead_time = POLICY_DICT[item].get("lead_time", 1)
    procurement_type = POLICY_DICT[item].get("procurement_type", "RAW")
    
    
    #STEP 0: Initialization : tạo khung item_df (index, col) ---> seeding (day0 on-hand , gross_demand [ind, dep], scheduled_receipt) 



    
    start_date, end_date = find_min_max_date(item_code = item, independent_demand_dict= demand_by_item, 
                                             global_dependent_dict = global_dependent_demand_dict, sr_dict= sr_by_item )
    my_index = date_range_index(start_date - pd.Timedelta(days=1), end_date)
    
    
    TP_MAP[item] = item_df_initializing(item_code = item, cal_cols= COLS_MRP_COMPUTE, date_index= my_index,
                                        on_hand = on_hand, independent_demand_dict= demand_by_item, 
                                        global_dependent_dict = global_dependent_demand_dict, sr_dict= sr_by_item
    )
    

    #STEP 1 : CALCULATION NET REQUIREMENT
    current_df = TP_MAP[item] #truy cap vao current DF of TP_MAP ---> xem xet có nên COPY hay không ????
    calculate_net_requirement(tp_item_df=current_df, cal_cols= COLS_MRP_COMPUTE )
    
    #STEP 2: FINDING LOT SIZE
    calculate_lot_size(tp_item_df=current_df, cal_cols=COLS_MRP_COMPUTE, 
                    safety_stock= ss_qty, policy_name= policy_name, policy_param=policy_param)

    #STEP 3: DEFINE PROCUREMENT TYPE AND SCHEDULING --> Planned Order
    
    ### --- PREPARE INPUT ---
    input_df = current_df.loc[:,"Lot size"].copy() #this is pd.series
    input_df = input_df.to_frame(name = "Lot size") #convert into pd.dataframe
    input_df = input_df[input_df["Lot size"] > 0]
    
    ### --- RUN logic ---
    tempt_planned_order_df = calculate_planned_order(item_name = item, tp_item_df= input_df, 
                                                procurement_type= procurement_type, lead_time= lead_time,
                                                today = TODAY, output_cols= COLS_PLANNED_ORDER)
    
    planned_order_list.append(tempt_planned_order_df)
    #STEP 4: EXPLODE QTY BASED on planned order and update the tp_df gross_requirement
    
    if procurement_type != "RAW": # run only when production order

        #--- get result ---
        tempt_exploded_dict, exploded_df = exploding_parent_item(item_name= item, item_planned_order_df= tempt_planned_order_df,
                                            bom_df = BOM_DF, scheduled_receipt= sr_by_item, lead_time= lead_time,
                                            input_cols= COLS_PLANNED_ORDER, output_cols = COLS_GROSS_REQUIREMENT)
        
        # --- write 
        #### for display: into list of df
        dependent_demand_list.append(exploded_df)
        #### for computation: into dict of (list of dict)
        for item, records in tempt_exploded_dict.items():
            global_dependent_demand_dict[item].extend(records)
            
            


#OUTPUT
planned_order_df = pd.concat(planned_order_list, ignore_index= True, sort = False)
dependent_demand_df = pd.concat(dependent_demand_list, ignore_index= True, sort = False)


In [ ]:
TP_MAP['E']

In [ ]:
POLICY_DICT['A']

## report/result writing

### 01 final demand dataframe

In [ ]:

independent_demand_df = demand_order_df.rename(columns={
    "date": "req_date",
    "qty": "gross_req_qty"
}).assign(
    source="Sales_Order",
    source_item=None,
    urgency_status=None,
    demand_type="Independent"
)

dependent_demand_df = dependent_demand_df.assign(demand_type = "Dependent")

final_demand_df = pd.concat([independent_demand_df, dependent_demand_df], ignore_index=True)

final_demand_df[final_demand_df['item'] == "E"]

### 02. MRP display

In [ ]:
BALANCE_COLS = ['Balance','Balance (+SR)', 'Balance(+SR+PR)', "Balance(+SR+1st PR)"] #FIX THU TU
SUM_COLS = ["Gross Req", "Scheduled Rcpt", 'planned_receipt', 'planned_release'] #FIX THU TU
HORIZON_MODE = ['Daily', 'Weekly'] #FIX THU TU
YOUR_MODE_CHOICE = "Daily"


target_cols1 = ['Gross Req', 'Scheduled Rcpt']
target_cols2 = ['item','planned_qty','receipt_date','release_date']

planned_order_per_item = dict(tuple(planned_order_df.groupby('item')))

#MAIN

list_result = []

for item in ITEMS_ORDERED:
    # 1. ------- INPUT------- 
    
    mrp_computed_df = TP_MAP[item].loc[:, target_cols1].copy()
    planned_df = planned_order_per_item[item].loc[:, target_cols2].copy()
    
    on_hand = float( ONHAND_DICT[item].get("qty", 0) )
    item_desc = ITEM_MASTER_DICT[item].get("desc", "")
    policy_param = POLICY_DICT[item]
    
    # 2. -------  Run MRP DISPLAY COMPUTATION — returns a *new* DataFrame ------- 
    result_df = step1_add_planned_col(input_mrp_df = mrp_computed_df,  input_planned_df = planned_df)

    result_df = step2_add_balance_col(input_mrp_df = result_df, on_hand = on_hand, bal_col = BALANCE_COLS, sum_col = SUM_COLS)

    result_df = step3_MRP_display(input_mrp_df = result_df, mode = YOUR_MODE_CHOICE,
                                      bal_col = BALANCE_COLS, sum_col = SUM_COLS, today = TODAY)
    
    # added_info_dict = dict(item_code = item, 
    #                                item_desc = item_desc , on_hand = on_hand, item_policy_param = policy_param )

    # result_df = step4_add_info(input_mrp_df = result_df, **added_info_dict)
    result_df = step4_add_info(input_mrp_df = result_df, item_code = item, 
                                   item_desc = item_desc , on_hand = on_hand, item_policy_param = policy_param )

    
    # 3. ------- Append to list of df for FINAL CONCAT -------------
    
    result_df['Item Code'] = item # Tag for traceability
    result_df = result_df.set_index('Item Code')
    list_result.append(result_df)
    

final_mrp_display_df = pd.concat(list_result, ignore_index= False)




#WRITING RESULT

# excel_path = "output_python_example/TAIDUC_MRP.xlsx"
# final_mrp_display_df.to_excel(excel_path, index = True)

    

In [ ]:
final_mrp_display_df

In [ ]:
# 02. MRP DISPLAYS

# excel_path = "output_python_example/TAIDUC_MRP.xlsx"
# with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
#     for item in ITEMS_ORDERED:
#         tempt_mrp_computed = TP_MAP[item].copy()
#         tempt_mrp_display = tempt_mrp_computed.T
        
#         tempt_mrp_display.to_excel(writer, sheet_name=item, index = True)
#         # combined

### 03. Order Recommendation report

In [ ]:
# structure: raw_data + 3 báo cáo

# - NHU CẦU MUA HÀNG/SX: (item + order_type) --> [qty, receipt_date]
#     - sliced by month, week , date
# - ACTION (đổ lại 1 tuần): order_type --> item + policy_name --> [qty, release_date]
# - URGENT ORDER = cần xuất gấp (order_type + item )

In [ ]:
planned_order_df

In [ ]:
planned_order_df["week"] = planned_order_df["receipt_date"].dt.strftime("W%U-%Y")

weekly_planned_df = (
    pd.pivot_table(
        planned_order_df,
        values="planned_qty",
        index=["item", "order_type"],
        columns="week",
        aggfunc="sum",
        fill_value=0
    )
    .sort_index(axis=1)
).reset_index()

weekly_planned_df

In [ ]:
urgent_report_df = (
    planned_order_df[planned_order_df['urgency_status'] == "urgent"]
    .groupby("item", as_index=False)
    .agg({"planned_qty": "sum"})
)

urgent_report_df

In [ ]:
from config import (PATH_MASTER, PATH_TRANSACTION, PATH_OUTPUT, 
                    FILE_ITEM, FILE_BOM, FILE_POLICY, FILE_DEMAND, FILE_SUPPLY, FILE_OH,
                    SOURCE_REQUIRED_COLS,
                    HORIZON_END_DAYS
)
import os
import pandas as pd

def _read_csv(folder, filename, parse_dates=None):
    """Internal helper for reading CSVs safely."""
    file_path = os.path.join(folder, filename)
    
    try:
        if filename.endswith(".txt"):
            df = pd.read_csv(file_path, sep="\t", parse_dates=parse_dates, encoding="utf-8-sig")
        else:
            
            df = pd.read_csv(file_path, parse_dates=parse_dates)
        print(f"[INFO] Loaded {filename:<20} → {len(df):>5,} rows")
        return df

    except FileNotFoundError:
        print(f"[WARN] File not found: {file_path}")
        return pd.DataFrame()
    except Exception as e:
        print(f"[ERROR] Failed to load {filename}: {e}")
        return pd.DataFrame()
    
    
    
a = _read_csv(PATH_TRANSACTION, FILE_DEMAND)







# MRP taiduc

In [ ]:
from helpers.helper import source_loader, writting_result, prepare_input
from main_script import mrp_computation, mrp_display


source_data = source_loader()
input_data = prepare_input(source_data)
result = mrp_computation(input_data)

# input_display = dict(list_of_df= input_data, mrp_result=result, raw_demand_df= source_data['transaction']['demand_orders'])
# display_result = mrp_display(**input_display)

# writting_result(data = display_result)





------------------Start Loading-------------------
[INFO] Loaded demand_orders.txt    →   669 rows
[INFO] Loaded onhand.txt           →   146 rows
[INFO] Loaded supply_orders.txt    →   164 rows
[INFO] Loaded bom_master.txt       →   404 rows
[INFO] Loaded item_master.txt      →   147 rows
[INFO] Loaded policy_master.txt    →   147 rows

[INFO] Validating column integrity...
[OK]   demand_orders   → Columns validated ✓
[OK]   onhand          → Columns validated ✓
[OK]   supply_orders   → Columns validated ✓
[OK]   bom_master      → Columns validated ✓
[OK]   item_master     → Columns validated ✓
[OK]   policy_master   → Columns validated ✓
[INFO] ✅ Input data loaded successfully.
List of ALL ITEM LIST , sorted by true bom level:
------FINAL_LV:  {'ENB-703': 0, 'ENB-705': 0, 'ENB-707': 0, 'GRB-003': 0, 'GRB-005': 0, 'GRB-007': 0, 'TAIDUC-001': 0, 'TAIDUC-002': 0, 'TRB-403': 0, 'TRB-405': 0, 'TRB-407': 0, 'XCB-103': 0, 'XCB-105': 0, 'XCB-107': 0, 'ACR-032': 1, 'ACR-034': 1, 'ACR-036': 1,

In [5]:
result['dependent_demand_df'].item_code.unique()

array(['TRF-403', 'ACS-165', 'AST-055', 'SPR-160', 'ACR-034', 'HBG-100',
       'PDL-200', 'BRK-202', 'BRK-204', 'HBM-750', 'CHN-512', 'TYR-427',
       'FRK-727', 'SHX-708', 'CSS-812', 'MEC-813', 'WLS-427', 'SDL-239',
       'ENF-707', 'SPR-213', 'ACR-032', 'BRK-402', 'BRK-404', 'HBM-800',
       'TYR-729', 'TYR-829', 'SHX-709', 'FRK-829', 'WLS-429', 'SPD-080',
       'CCS-170', 'CST-050', 'ACR-038', 'GRK-000', 'GRF-003', 'GTP-100',
       'HBG-420', 'CHN-513', 'SDL-147', 'TYR-170', 'BRK-134', 'BRK-135',
       'CSS-912', 'MEC-913', 'WLS-723', 'GRF-007', 'HBG-460', 'XCF-105',
       'AST-070', 'ACR-036', 'HBM-740', 'TYR-229', 'FRK-429', 'SHX-410',
       'WLS-127', 'TRF-405', 'SPR-185', 'TYR-429', 'FRK-729', 'XCF-107',
       'GRF-005', 'HBG-440', 'TRF-407', 'XCF-103', 'ENF-705', 'CHN-511',
       'ENF-703', 'TYR-727', 'TYR-827', 'ASL-140', 'ABB-073', 'PCC-101',
       'CPK-145', 'CPK-240', 'CPK-440', 'DCL-140', 'GRS-140', 'MIB-111',
       'PBK-140', 'HSX-214', 'FHB-932', 'RHB-932', 

# ARCHIEVE

### OLD STEP 1

In [ ]:
## using tp_map + planned_order_df + current inventory on-hand as input


# ------ Prepare INPUT , OUTPUT init--------- 
mrp_df_by_item = TP_MAP['E'][['Gross Req','Scheduled Rcpt',]].copy()

item_groups = dict(tuple(planned_order_df.groupby('item')))
planned_order_by_item = item_groups['E'][['item','planned_qty','receipt_date','release_date']].copy()


# ---------- reindex the main dataframe to include all dates ----------


all_dates = (mrp_df_by_item.index
             .union(planned_order_by_item['receipt_date'])
             .union(planned_order_by_item['release_date'])
            ).sort_values()

mrp_df_by_item = mrp_df_by_item.reindex(
    pd.date_range( all_dates.min(), all_dates.max(), freq='D' )
)


# --- aggregate planned qty by release/receipt date then turn into dict ---

release_map = (
    planned_order_by_item.groupby("release_date")["planned_qty"]
    .sum().to_dict()
)
receipt_map = (
    planned_order_by_item.groupby("receipt_date")["planned_qty"]
    .sum().to_dict()
)

# --- map value of the columns to index ---


mrp_df_by_item['planned_receipt'] = 0
mrp_df_by_item['planned_release'] = 0

mrp_df_by_item['planned_receipt'] = mrp_df_by_item.index.to_series().map(receipt_map)
mrp_df_by_item['planned_release'] = mrp_df_by_item.index.to_series().map(release_map)


# --- fill na value and sorting ---
mrp_df_by_item = mrp_df_by_item.fillna(0).sort_index().astype(int)



mrp_df_by_item

### OLD STEP 2

In [ ]:
BALANCE_COLS = ['Balance','Balance (+SR)', 'Balance(+SR+PR)', "Balance(+SR+1st PR)"]
SUM_COLS = ["Gross Req", "Scheduled Rcpt", 'planned_release', 'planned_receipt']
on_hand_E = ONHAND_DICT['E']['qty']


mrp_df_by_item.loc[mrp_df_by_item.index[0], BALANCE_COLS] = on_hand_E #mrp_df_by_item.at[mrp_df_by_item.index[0], 'Balance'] = on_hand_E JUST SCALAR

# Identify first planned receipt date (if any)

first_pr_date = (
    mrp_df_by_item.loc[mrp_df_by_item['planned_receipt'] > 0].index.min()
    if (mrp_df_by_item['planned_receipt'] > 0).any()
    else None
)

# CACH 1 Iterate from 2nd row onward

for i in range(1, len(mrp_df_by_item)):
    
    # Define row and index
    prev_row = mrp_df_by_item.iloc[i - 1]
    curr_row = mrp_df_by_item.iloc[i]
    curr_idx = mrp_df_by_item.index[i]
    # Find component value
    gross = curr_row['Gross Req']
    sched = curr_row['Scheduled Rcpt']
    plan_r = curr_row['planned_receipt']
    
    add_first_pr = plan_r if curr_idx == first_pr_date else 0
    # Compute balance
    mrp_df_by_item.at[curr_idx, 'Balance'] = prev_row['Balance'] - gross
    mrp_df_by_item.at[curr_idx, 'Balance (+SR)'] = prev_row['Balance (+SR)'] + sched - gross
    mrp_df_by_item.at[curr_idx, 'Balance(+SR+PR)'] = prev_row['Balance(+SR+PR)'] + sched + plan_r - gross
    mrp_df_by_item.at[curr_idx, 'Balance(+SR+1st PR)'] = prev_row['Balance(+SR+1st PR)'] + sched + add_first_pr - gross

# CACH 2 
    # df['first_PR_only'] = 0
    # if first_pr_date is not None:
    #     df.loc[first_pr_date, 'first_PR_only'] = df.loc[first_pr_date, 'planned_receipt']

    # df['Balance'] = on_hand - df['Gross Req'].cumsum()
    # df['Balance (+SR)'] = on_hand + (df['Scheduled Rcpt'] - df['Gross Req']).cumsum()
    # df['Balance(+SR+PR)'] = on_hand + (df['Scheduled Rcpt'] + df['planned_receipt'] - df['Gross Req']).cumsum()
    # df['Balance(+SR+1st PR)'] = on_hand + (df['Scheduled Rcpt'] + df['first_PR_only'] - df['Gross Req']).cumsum()  
    
mrp_df_by_item
    

### OLD STEP 3

In [ ]:
# Aggregate overdue part
agg_dict = {**{c: 'sum' for c in SUM_COLS},
            **{c: 'last' for c in BALANCE_COLS}}

df_before = mrp_df_by_item[mrp_df_by_item.index < TODAY].copy()
df_after = mrp_df_by_item[mrp_df_by_item.index >= TODAY].copy()
df_after_week = df_after.copy()


## HANDLE BEFORE
df_before['final_index'] = 'overdue'

df_before = (
    df_before
    .groupby('final_index', as_index=True)
    .agg(agg_dict)
)
df_before.index.name = None
## HANDLE AFTER DAILY

iso = df_after.index.isocalendar()

df_after['final_index'] = [
    # Condition 1,2 : Monday or first index (even if not Monday) → add week marker
    f"{d.strftime('%d/%m')} ,, W{w:02d}Y{y % 100}" if d.weekday() == 0 or d == df_after.index[0]
    else f"{d.strftime('%d/%m')} ,, W{w:02d}Y{y % 100}" if d == df_after.index[0]
    # Condition 3: other days → plain date
    else d.strftime('%d/%m')
    for d, w, y in zip(df_after.index, iso.week, iso.year) #BẢN CHẤT VẪN LÀ TẠO COLUMNS
]

df_after = df_after.set_index('final_index')
df_after.index.name = None


## HANDLING AFTER WEEKLY

df_after_week['week_number'] = df_after_week.index.to_series().dt.isocalendar().week
df_after_week['year'] = df_after_week.index.to_series().dt.year % 100

df_after_week['week_label'] = (
    "W" + df_after_week['week_number'].astype(int).astype(str).str.zfill(2)
    + "Y" + df_after_week['year'].astype(int).astype(str)
)


df_after_week = (
    df_after_week
    .groupby('week_label', as_index=True)
    .agg(agg_dict)
)
df_after_week.index.name = None


## APPENDING

final_mrp_df_by_item = pd.concat([df_before, df_after])
final_mrp_df_by_item = final_mrp_df_by_item.T
final_mrp_df_by_item = final_mrp_df_by_item.reset_index().rename(columns={'index': 'MRP_element'})
final_mrp_df_by_item


### OLD STEP 4

In [ ]:
info_dict = {'item_code': "E",
             'item_desc': "Toyoto BM 100",
             'current_stock': 15000,
             'lead_time': 7,
             'safety_stock': 300,
             'order_policy': "L4L",
             'rounding_value': "5000",
}
# Convert dict → DataFrame

info_df = pd.DataFrame(list(info_dict.items()), columns=['item_code_info', 'value'])
    
# Find max row count
max_len = max(len(info_df), len(final_mrp_df_by_item))

# Pad info_df and final_mrp
info_df = info_df.reindex(range(max_len)).fillna('')
final_mrp_df_by_item = final_mrp_df_by_item.reindex(range(max_len)).fillna('')

final_mrp_df_by_item = pd.concat([info_df, final_mrp_df_by_item], axis=1)

final_mrp_df_by_item['item_code_index'] = info_dict['item_code']
final_mrp_df_by_item = final_mrp_df_by_item.set_index('item_code_index')
final_mrp_df_by_item.index.name = None

In [ ]:
final_mrp_df_by_item

## zzzzzzzzzzzzzzzzz Mistake holder

In [ ]:
def mistake_holder(): #do not touch
    #BAD WAYS FOR INITILIZATION
    #STEP 0. INITIALIZATION
    ## create tp_map, df, and inventory
    # for it in items:
    #     df = pd.DataFrame(0, index=idx, columns=COLS1)
    #     df.at[idx[0], "POH 1 (before LS)"] = int(on_hand0.get(it, 0))
    #     df.at[idx[0], "POH 2 (after LS)"] = int(on_hand0.get(it, 0))
    #     tp_map[it] = df

    # # Seed gross requirements from demand_records (MPS)
    # for rec in demand_records:
    #     it = rec["item"]
    #     d = rec["date"]
    #     if d in tp_map[it].index:
    #         tp_map[it].at[d, "Gross Req"] += int(rec["qty"])
    # # seed scheduled receipts
    # for _, r in scheduled_receipts.iterrows():
    #     d = r["date"]
    #     it = r["item"]
    #     if d in tp_map[it].index:
    #         tp_map[it].loc[d, "Scheduled Rcpt"] += int(r["qty"])
    pass

In [ ]:
# DURING THE LOOP in item in ITEM_ORDERED:

# first_date_independent = min( (rec['date'] for rec in demand_by_item.get(item,[]) ),
#                              default=pd.Timestamp.max
# )
# first_date_dependent = min( (rec['req_date'] for rec in global_dependent_demand_dict.get(item,[]) ),
#                              default=pd.Timestamp.max
# )
# first_date_sr = min( (rec['date'] for rec in sr_by_item.get(item,[]) ),
#                              default=pd.Timestamp.max
# )  
# last_date_independent = max( (rec['date'] for rec in demand_by_item.get(item,[]) ),
#                              default=pd.Timestamp.min
# )
# last_date_dependent = max( (rec['req_date'] for rec in global_dependent_demand_dict.get(item,[]) ),
#                              default=pd.Timestamp.min
# )
# last_date_sr = max( (rec['date'] for rec in sr_by_item.get(item,[]) ),
#                              default=pd.Timestamp.min
# )  
# start_date = min(first_date_independent, first_date_dependent, first_date_sr)
# end_date = max(last_date_independent, last_date_dependent,  last_date_sr)

In [ ]:
"""SEEDING no function"""


    # ## create df in {item : pd.df} and seeding inventory
    # df = pd.DataFrame(0, index=my_index, columns=COLS_MRP_COMPUTE)
    # df.at[my_index[0], "POH 1 (before LS)"] = on_hand
    # df.at[my_index[0], "POH 2 (after LS)"] = on_hand
    
    # TP_MAP[item] = df
    
    # ## seeding GROSS INDENPEDENT DEMAND
    # for rec in demand_by_item.get(item, []):
    #     d = rec["date"]
    #     qty = rec["qty"]
    #     if d in TP_MAP[item].index:
    #         TP_MAP[item].at[d, "Gross Req"] += int(qty)
    # ## seeding GROSS DEDENDENT DEMAND
    
    # for rec in global_dependent_demand_dict.get(item, []):
    #     d = rec["req_date"]
    #     qty = rec["gross_req_qty"]
    #     if d in TP_MAP[item].index:
    #         TP_MAP[item].at[d, "Gross Req"] += int(qty)
    
    # ## seeding SCHEDULE RECEIPT
    # for rec in sr_by_item.get(item, []):
    #     d = rec["date"]
    #     qty = rec["qty"]
    #     if d in TP_MAP[item].index:
    #         TP_MAP[item].at[d, "Scheduled Rcpt"] += int(qty)

## defense mechanism for step 1 of MRP display


In [ ]:
import pandas as pd

target_cols1 = ['Gross Req', 'Scheduled Rcpt']
target_cols2 = ['item','planned_qty','receipt_date','release_date']

planned_order_per_item = dict(tuple(planned_order_df.groupby('item')))
list_result = []

for item in ITEMS_ORDERED:
    # --- Step 0: Defensive guards ---
    has_mrp = item in TP_MAP and not TP_MAP[item].empty
    has_plan = item in planned_order_per_item and not planned_order_per_item[item].empty
    
    if not has_mrp and not has_plan:
        # 🧩 both missing → blank df placeholder
        empty_df = pd.DataFrame(columns=target_cols1 + ['planned_receipt', 'planned_release'])
        empty_df['Item Code'] = item
        list_result.append(empty_df)
        continue

    # --- Step 1: Extract only required input columns ---
    if has_mrp:
        mrp_computed_df = TP_MAP[item].loc[:, target_cols1].copy()
    else:
        mrp_computed_df = pd.DataFrame(columns=target_cols1)

    if has_plan:
        planned_df = planned_order_per_item[item].loc[:, target_cols2].copy()
    else:
        planned_df = pd.DataFrame(columns=target_cols2)

    # --- Step 2: Run computation safely ---
    try:
        result_df = step1_add_planned_col(
            input_mrp_df=mrp_computed_df,
            input_planned_df=planned_df
        )
    except Exception as e:
        # ⚠️ Defensive fallback if computation fails for this item
        print(f"[WARN] Step1 computation failed for item {item}: {e}")
        result_df = pd.DataFrame(columns=target_cols1 + ['planned_receipt', 'planned_release'])

    # --- Step 3: Tag and append ---
    result_df['Item Code'] = item
    list_result.append(result_df)

# --- Step 4: Final concatenation ---
final_mrp_display_df = pd.concat(list_result, ignore_index=False)


In [12]:
import pandas as pd
ts = pd.Timestamp('2025-11-25')


new_ts = ts + pd.offsets.Week(weekday=3,n=1)

new_ts


Timestamp('2025-11-27 00:00:00')